## Purpose:
    This script preprocesses raw GPS trajectory data for Jeju Island and converts
    the cleaned GPS trajectories into hexagon-based movement sequences.

## Input:
    - Raw GPS trajectory CSV files for the training and validation datasets.
    - Jeju boundary shapefile used to retain GPS points located within Jeju Island.
    - Hexagon shapefile used to assign each GPS point to a spatial cell.

## Output:
    - Split GPS trajectory files containing only valid records within Jeju Island.
    - Hexagon-based trajectory CSV files containing cell IDs, stay durations, and timestamps.

### (actual processing is parallelized in a separate .py script)

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import os.path as osp
from tqdm import tqdm
from multiprocessing import get_context
import re
import warnings
warnings.filterwarnings("ignore", message="Geometry is in a geographic CRS*")

## Clip only the features within Jeju Island.

### Training Dataset

In [2]:
IN_DIR   = "./data/sample_raw_data/Jeju_data/Training/gps"
OUT_DIR  = "./data/sample_processed_inputs/Training/gps_split_jeju"
JEJU_SHP = "./data/jeju_shp/jeju.shp"
USE_PARALLEL = True
N_WORKERS    = 20

os.makedirs(OUT_DIR, exist_ok=True)

def valid_mask_from_xy(df: pd.DataFrame, jeju_union) -> pd.Series:
    if "X_COORD" not in df.columns or "Y_COORD" not in df.columns:
        return pd.Series([False] * len(df))
    x = pd.to_numeric(df["X_COORD"], errors="coerce")
    y = pd.to_numeric(df["Y_COORD"], errors="coerce")
    finite_mask = x.notna() & y.notna() & np.isfinite(x) & np.isfinite(y)
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(x, y), crs="EPSG:4326")
    inside_mask = gdf.geometry.intersects(jeju_union)
    return (finite_mask & inside_mask).fillna(False)

def split_and_save(df: pd.DataFrame, valid_mask: pd.Series, out_base_path: str, travel_id: str):
    if valid_mask.any():
        block_id = (valid_mask & ~valid_mask.shift(fill_value=False)).cumsum()
        groups = df[valid_mask].groupby(block_id[valid_mask], sort=True)
        cnt = 0
        for k, sub in enumerate(groups, start=1):
            _, subdf = sub
            if "TRAVEL_ID" not in subdf.columns:
                subdf = subdf.copy(); subdf["TRAVEL_ID"] = travel_id
            subdf.to_csv(f"{out_base_path}_{k}.csv", index=False)
            cnt += 1
        return cnt
    else:
        pd.DataFrame({"TRAVEL_ID":[travel_id], "NODE_ID":[-1]}).to_csv(f"{out_base_path}_none.csv", index=False)
        return 0

def process_one_file(args):
    filename, jeju_union = args
    try:
        in_path  = osp.join(IN_DIR, filename)
        base, _  = osp.splitext(filename)
        out_base = osp.join(OUT_DIR, base)

        df = pd.read_csv(in_path)
        travel_id = df["TRAVEL_ID"].iloc[0] if "TRAVEL_ID" in df.columns else base

        valid_mask = valid_mask_from_xy(df, jeju_union)
        saved = split_and_save(df, valid_mask, out_base, travel_id)
        return filename, saved, (saved == 0)
    except Exception as e:
        return filename, -1, str(e)

def main():
    gdf_jeju  = gpd.read_file(JEJU_SHP).to_crs(epsg=4326)
    jeju_union = gdf_jeju.buffer(0.005).unary_union

    file_list = [f for f in os.listdir(IN_DIR) if f.lower().endswith(".csv")]
    results = []

    if USE_PARALLEL:
        try:
            ctx = get_context("fork")
        except ValueError:
            ctx = get_context("spawn")
        with ctx.Pool(processes=N_WORKERS) as pool:
            it = pool.imap_unordered(process_one_file, [(f, jeju_union) for f in file_list])
            for res in tqdm(it, total=len(file_list), desc="Processing (parallel)"):
                results.append(res)
    else:
        for f in tqdm(file_list, desc="Processing (sequential)"):
            results.append(process_one_file((f, jeju_union)))

    total = len(results)
    ok    = sum(1 for _, c, nf in results if c >= 0 and not nf)
    none  = sum(1 for _, c, nf in results if c == 0)
    fail  = [r for r in results if r[1] == -1]

    print(f"\n=== Summary ===")
    print(f"Total files     : {total}")
    print(f"Saved (>=1 part): {ok}")
    print(f"Only -1 (_none) : {none}")
    print(f"Failed          : {len(fail)}")
    if fail:
        for fn, _, msg in fail[:10]:
            print(f" - {fn}: {msg}")

if __name__ == "__main__":
    main()


Processing (parallel): 100%|██████████| 256/256 [01:25<00:00,  3.00it/s]


=== Summary ===
Total files     : 256
Saved (>=1 part): 152
Only -1 (_none) : 104
Failed          : 0


### Valdiation Dataset (Same code with Training dataset)

In [3]:
IN_DIR   = "./data/sample_raw_data/Jeju_data/Validation/gps"
OUT_DIR  = "./data/sample_processed_inputs/Validation/gps_split_jeju"
JEJU_SHP = "./data/jeju_shp/jeju.shp"
USE_PARALLEL = True
N_WORKERS    = 20

os.makedirs(OUT_DIR, exist_ok=True)

def valid_mask_from_xy(df: pd.DataFrame, jeju_union) -> pd.Series:
    if "X_COORD" not in df.columns or "Y_COORD" not in df.columns:
        return pd.Series([False] * len(df))
    x = pd.to_numeric(df["X_COORD"], errors="coerce")
    y = pd.to_numeric(df["Y_COORD"], errors="coerce")
    finite_mask = x.notna() & y.notna() & np.isfinite(x) & np.isfinite(y)
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(x, y), crs="EPSG:4326")
    inside_mask = gdf.geometry.intersects(jeju_union)
    return (finite_mask & inside_mask).fillna(False)

def split_and_save(df: pd.DataFrame, valid_mask: pd.Series, out_base_path: str, travel_id: str):
    if valid_mask.any():
        block_id = (valid_mask & ~valid_mask.shift(fill_value=False)).cumsum()
        groups = df[valid_mask].groupby(block_id[valid_mask], sort=True)
        cnt = 0
        for k, sub in enumerate(groups, start=1):
            _, subdf = sub
            if "TRAVEL_ID" not in subdf.columns:
                subdf = subdf.copy(); subdf["TRAVEL_ID"] = travel_id
            subdf.to_csv(f"{out_base_path}_{k}.csv", index=False)
            cnt += 1
        return cnt
    else:
        pd.DataFrame({"TRAVEL_ID":[travel_id], "NODE_ID":[-1]}).to_csv(f"{out_base_path}_none.csv", index=False)
        return 0

def process_one_file(args):
    filename, jeju_union = args
    try:
        in_path  = osp.join(IN_DIR, filename)
        base, _  = osp.splitext(filename)
        out_base = osp.join(OUT_DIR, base)

        df = pd.read_csv(in_path)
        travel_id = df["TRAVEL_ID"].iloc[0] if "TRAVEL_ID" in df.columns else base

        valid_mask = valid_mask_from_xy(df, jeju_union)
        saved = split_and_save(df, valid_mask, out_base, travel_id)
        return filename, saved, (saved == 0)
    except Exception as e:
        return filename, -1, str(e)

def main():
    # Load the Jeju boundary inside the main workflow.
    gdf_jeju  = gpd.read_file(JEJU_SHP).to_crs(epsg=4326)
    jeju_union = gdf_jeju.buffer(0.005).unary_union

    file_list = [f for f in os.listdir(IN_DIR) if f.lower().endswith(".csv")]
    results = []

    if USE_PARALLEL:
        # Use fork on Linux if available; otherwise, use spawn.
        try:
            ctx = get_context("fork")
        except ValueError:
            ctx = get_context("spawn")
        with ctx.Pool(processes=N_WORKERS) as pool:
            it = pool.imap_unordered(process_one_file, [(f, jeju_union) for f in file_list])
            for res in tqdm(it, total=len(file_list), desc="Processing (parallel)"):
                results.append(res)
    else:
        for f in tqdm(file_list, desc="Processing (sequential)"):
            results.append(process_one_file((f, jeju_union)))

    total = len(results)
    ok    = sum(1 for _, c, nf in results if c >= 0 and not nf)
    none  = sum(1 for _, c, nf in results if c == 0)
    fail  = [r for r in results if r[1] == -1]

    print(f"\n=== Summary ===")
    print(f"Total files     : {total}")
    print(f"Saved (>=1 part): {ok}")
    print(f"Only -1 (_none) : {none}")
    print(f"Failed          : {len(fail)}")
    if fail:
        for fn, _, msg in fail[:10]:
            print(f" - {fn}: {msg}")

if __name__ == "__main__":
    main()


Processing (parallel): 100%|██████████| 32/32 [00:21<00:00,  1.46it/s]


=== Summary ===
Total files     : 32
Saved (>=1 part): 19
Only -1 (_none) : 13
Failed          : 0


##  Convert a GPS trajectory segment into a hexagon-based movement sequence.

### This function assigns each GPS point to a hexagon cell using a spatial join. GPS points that are not located within any hexagon cell are assigned to the nearest hexagon cell using nearest-neighbor spatial join. After cell assignment, consecutive records with the same hexagon ID are compressed into transition points, and the stay duration in each hexagon cell is calculated based on the timestamp difference between consecutive transitions.

In [4]:
def make_hexa_csv_series(file_df: pd.DataFrame,
                         save_dir: str,
                         base_name: str,
                         hex_gdf: gpd.GeoDataFrame):
    
    os.makedirs(save_dir, exist_ok=True)

    # Step 0: Apply preliminary filters.
    if len(file_df) <= 2:
        return

    #  # Skip files out of jeju.
    if ("X_COORD" not in file_df.columns) or ("Y_COORD" not in file_df.columns):
        return
    if (file_df["X_COORD"].eq(-1).any()) or (file_df["Y_COORD"].eq(-1).any()):
        return

    # Step 1: Convert GPS records into a GeoDataFrame.
    gdf_gps = gpd.GeoDataFrame(
        file_df,
        geometry=gpd.points_from_xy(file_df["X_COORD"], file_df["Y_COORD"]),
        crs="EPSG:4326"
    ).to_crs(hex_gdf.crs)

    # Step 2: Perform within join.
    joined = gpd.sjoin(
        gdf_gps,
        hex_gdf[["new_id", "geometry"]],
        how="left",
        predicate="within"
    ).drop(columns=["index_right"])

    # Step 3: Fill missing values with nearest hexagons.
    miss_mask = joined["new_id"].isna()
    if miss_mask.any():
        nearest = gpd.sjoin_nearest(
            gdf_gps.loc[miss_mask, ["geometry"]],
            hex_gdf[["new_id", "geometry"]],
            how="left",
            distance_col="dist_near"
        ).drop(columns=["index_right"])
        # 벡터 치환
        joined.loc[miss_mask, "new_id"] = nearest["new_id"].to_numpy()

    # Step 4: Create hexa_code.
    if joined["new_id"].isna().all():
        return

    joined = joined.drop(columns="geometry")
    joined["hexa_code"] = joined["new_id"].astype("Int64")  # nullable integer type
    joined = joined.drop(columns="new_id")

    # Step 5: Extract consecutive change intervals
    hexa = joined["hexa_code"]
    if hexa.isna().any():
        joined = joined[hexa.notna()]
        hexa = joined["hexa_code"]
        if len(hexa) == 0:
            return

    change_mask = hexa.ne(hexa.shift(fill_value=hexa.iloc[0]))
    if not change_mask.any():
        return

    changed = joined.loc[change_mask, ["hexa_code", "DT_MIN"]].copy()

    # Calculate stay duration in seconds until the next transition.
    t = pd.to_datetime(changed["DT_MIN"])
    t_next = t.shift(-1, fill_value=t.iloc[-1])
    time_diff_sec = (t_next.to_numpy() - t.to_numpy()) / np.timedelta64(1, "s")

    # Step 6: Construct the output dataframe.
    out = pd.DataFrame({
        "hexa_code": changed["hexa_code"].astype(int).to_numpy(),
        "time_diff_sec": time_diff_sec.astype(float),
        "time": changed["DT_MIN"].to_numpy()
    })

    if "MOBILE_NUM_ID" in joined.columns:
        out["MOBILE_NUM_ID"] = joined["MOBILE_NUM_ID"].iloc[0]
    if "TRAVEL_ID" in joined.columns:
        out["TRAVEL_ID"] = joined["TRAVEL_ID"].iloc[0]

    out_path = os.path.join(save_dir, f"hexa_{base_name}")
    out.to_csv(out_path, index=False, encoding="utf-8-sig")

In [5]:
jeju_hexa_path = "./data/new_hexagraph/hexa_network_with_road.shp"
gdf = gpd.read_file(jeju_hexa_path)

def numerical_sort_key(file):
    return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', file)]

# data for training
folder_Tr = './data/sample_processed_inputs/Training/gps_split_jeju/'
route_csv_list_Tr = sorted(
    [f for f in os.listdir(folder_Tr) if f.endswith(".csv")],
    key=numerical_sort_key
)

for i in range(len(route_csv_list_Tr)):
    f1 = pd.read_csv("./data/sample_processed_inputs/Training/gps_split_jeju/" + route_csv_list_Tr[i])
    make_hexa_csv_series(f1, save_dir = "./data/sample_processed_inputs/Training/gps_hexa/", base_name = route_csv_list_Tr[i], hex_gdf = gdf)
    
    if i%10 == 0:
        print(f'{i}/{len(route_csv_list_Tr)}')


# data for validation
folder_Va = './data/sample_processed_inputs/Validation/gps_split_jeju/'
route_csv_list_Va = sorted(
    [f for f in os.listdir(folder_Va) if f.endswith(".csv")],
    key=numerical_sort_key
)

for i in range(len(route_csv_list_Va)):
    f1 = pd.read_csv("./data/sample_processed_inputs/Validation/gps_split_jeju/"+route_csv_list_Va[i])
    make_hexa_csv_series(f1, save_dir = "./data/sample_processed_inputs/Validation/gps_hexa/", base_name = route_csv_list_Va[i], hex_gdf = gdf)
    
    if i%10 == 0:
        print(f'{i}/{len(route_csv_list_Va)}')

0/392
10/392
20/392
30/392
40/392
50/392
60/392
70/392
80/392
90/392
100/392
110/392
120/392
130/392
140/392
150/392
160/392
170/392
180/392
190/392
200/392
210/392
220/392
230/392
240/392
250/392
260/392
270/392
280/392
290/392
300/392
310/392
320/392
330/392
340/392
350/392
360/392
370/392
380/392
390/392
0/42
10/42
20/42
30/42
40/42
